In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/drive/MyDrive/Colab\ Notebooks/sorter/* .
!unzip -q processed.zip

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time, bisect, re, os

num_cores = os.cpu_count()
print(f"Num cores: {num_cores}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def split_identifier(identifier: str) -> str:
    s = re.sub(r"[_\-.:]+", " ", identifier.strip())
    s = re.sub(r'(?<=[a-z0-9])([A-Z])', r' \1', s)
    s = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', s)
    s = s.strip(" _")
    return s.lower()

class CharPairwiseDataset(Dataset):
    vocab = { c: i for i, c in enumerate('$abcdefghijklmnopqrstuvwxyz 0123456789', 1) }
    vocab['<pad>'] = 0

    def __init__(self, sentences: list[list[str]], max_len: int):
        self.sentences: list[list[str]] = [[split_identifier(w) for w in s] for s in sentences]
        self.max_len = max_len

        self.cum_len = []
        for words in self.sentences:
            l = len(words)
            self.cum_len.append(l * (l - 1))
        for i in range(1, len(self.cum_len)):
            self.cum_len[i] += self.cum_len[i - 1]

    @staticmethod
    def tokenize(word: str, max_len: int) -> torch.Tensor:
        tensor = torch.tensor([CharPairwiseDataset.vocab.get(c) for c in word[:max_len]], dtype=torch.int)
        if tensor.size(0) < max_len:
            tensor = torch.nn.functional.pad(tensor, (0, max_len - tensor.size(0)))
        return tensor

    def __len__(self):
        return self.cum_len[-1] if self.cum_len else 0

    def __getitem__(self, idx) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        sentence_idx = bisect.bisect_right(self.cum_len, idx)
        prev_cum = self.cum_len[sentence_idx - 1] if sentence_idx > 0 else 0
        this_idx = idx - prev_cum

        words = self.sentences[sentence_idx]
        l = len(words)
        i = this_idx // (l - 1)
        j = this_idx % (l - 1)
        if j >= i:
            j += 1

        token1 = self.tokenize(words[i], self.max_len)
        token2 = self.tokenize(words[j], self.max_len)
        target = 1 if i>j else 0
        return token1, token2, torch.tensor(target, dtype=torch.float)

In [ ]:
class CharEncoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_dim: int,
    ):
        super(CharEncoder, self).__init__()
        self.out_dim = hidden_dim * 2

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2,
        )
        self.fc = nn.Linear(hidden_dim * 2, self.out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(x)
        _, (h_n, _) = self.lstm(embedded)
        h_n = torch.cat((h_n[-2], h_n[-1]), dim=1)
        return self.fc(h_n)

In [ ]:
# Used for training
class CharPairwiseModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_dim: int,
    ):
        super(CharPairwiseModel, self).__init__()
        self.char_encoder = CharEncoder(vocab_size, embedding_dim, hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(self.char_encoder.out_dim * 4, self.char_encoder.out_dim * 2),
            nn.ReLU(self.char_encoder.out_dim * 2),
            nn.Dropout(0.2),
            nn.Linear(self.char_encoder.out_dim * 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        emb1 = self.char_encoder(x1)
        emb2 = self.char_encoder(x2)
        prod = emb1 * emb2
        diff = emb1 - emb2
        feature = torch.cat([emb1, emb2, prod, diff], dim=1)
        return self.classifier(feature).squeeze()

In [ ]:
# Used for building the ONNX model
class CharPairwiseModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_dim: int,
    ):
        super(CharPairwiseModel, self).__init__()
        self.char_encoder = CharEncoder(vocab_size, embedding_dim, hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(self.char_encoder.out_dim * 4, self.char_encoder.out_dim * 2),
            nn.ReLU(self.char_encoder.out_dim * 2),
            nn.Dropout(0.2),
            nn.Linear(self.char_encoder.out_dim * 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        emb1 = self.char_encoder(x1)
        emb2 = self.char_encoder(x2)
        prod = emb1 * emb2
        diff = emb1 - emb2
        feature1 = torch.cat([emb1, emb2, prod, diff], dim=1)
        feature2 = torch.cat([emb2, emb1, prod, -diff], dim=1)
        return (self.classifier(feature1).squeeze(), self.classifier(feature2).squeeze())

In [ ]:
def eval_test(model: CharPairwiseModel, dataloader: DataLoader, verbose=False):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x1, x2, target in dataloader:
            output = model(x1.to(device), x2.to(device))
            pred = (output >= 0.5).float()
            correct += (pred == target.to(device)).sum().item()
            total += target.size(0)
    return correct / total

def train(
    model: CharPairwiseModel,
    train_dataset: CharPairwiseDataset,
    test_dataset: CharPairwiseDataset,
    epochs: int = 10,
    batch_size: int = 2,
    lr: float = 0.001,
    patience: int = 5
):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=num_cores, pin_memory=True, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=num_cores, pin_memory=True, shuffle=False)

    criterion = nn.BCELoss()
    optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    # scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=patience-1, T_mult=2, eta_min=lr/100)

    global best_test_acc
    # best_test_acc = 0.0
    patience_counter = 0

    start = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch_idx, (x1, x2, target) in enumerate(train_loader):
            optimizer.zero_grad(set_to_none=True)
            output = model(x1.to(device), x2.to(device))
            loss = criterion(output, target.to(device))
            loss.backward()
            optimizer.step()
            # scheduler.step(epoch + batch_idx / len(train_loader))
            total_loss += loss.item() * x1.size(0)

        avg_loss = total_loss / len(train_dataset)
        test_acc = eval_test(model, test_loader)
        scheduler.step(test_acc)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Test Acc: {test_acc:.4f}, lr: {scheduler.get_last_lr()[0]}")

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            patience_counter = 0
            torch.save(model.state_dict(), "best_model-lstm-nobln.pth")
            print(f"Saved best model at epoch {epoch+1}")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    print(f"Training time: {time.time() - start:.2f}s")

In [ ]:
def evaluate(model: CharPairwiseModel, word1: str, word2: str):
    model.eval()
    token1 = CharPairwiseDataset.tokenize(split_identifier(word1), 0).unsqueeze(0).to(device)
    token2 = CharPairwiseDataset.tokenize(split_identifier(word2), 0).unsqueeze(0).to(device)
    with torch.no_grad():
        prob = model(token1, token2).item()
    prediction = 1 if prob >= 0 else -1
    return prediction, prob

In [ ]:
# This function was AI-generated
def load_processed_props(processed_folder):
    """
    Walk through the processed folder recursively.
    For each file, for each line, split the line at commas and strip extra whitespace.
    Returns a list of lists, where each inner list is the list of props from one line.
    """
    all_props = []  # This will hold our list of lists
    for root, dirs, files in os.walk(processed_folder):
        for file in files:
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    # Remove any leading/trailing whitespace and newlines
                    line = line.strip()
                    if line:  # skip empty lines
                        # Split by comma and remove extra whitespace around each prop
                        props = [prop.strip() for prop in line.split(',') if prop.strip()]
                        if file == 'final_order.txt':
                            final_order = props
                        else:
                            all_props.append(props)
    return all_props, final_order

if __name__ == "__main__":
    processed_folder = "processed"  # Adjust this path if needed.
    props_list, final_order = load_processed_props(processed_folder)
    print(len(props_list), len(final_order))

In [ ]:
model = CharPairwiseModel(len(CharPairwiseDataset.vocab), 16, 32).to(device)
print(sum(p.numel() for p in model.parameters()))
model

CharPairwiseModel(
  (char_encoder): CharEncoder(
    (embedding): Embedding(39, 16, padding_idx=0)
    (lstm): LSTM(16, 32, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
    (fc): Linear(in_features=64, out_features=64, bias=True)
  )
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU(inplace=True)
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=1, bias=True)
    (4): Sigmoid()
  )
)

In [ ]:
from sklearn.model_selection import train_test_split

train_props, test_props = train_test_split(props_list, test_size=0.2)
train_props.append(final_order)
# train_props = [final_order]
print(len(train_props), len(test_props))

train_dataset = CharPairwiseDataset(train_props, 25)
# test_dataset = train_dataset
test_dataset = CharPairwiseDataset(test_props, 25)
final_order_dataset = CharPairwiseDataset([final_order], 25)

In [ ]:
state_dict = torch.load("best_model-lstm-nobln.pth", map_location=device)
model.load_state_dict(state_dict, strict=False)
model

In [ ]:
best_test_acc = eval_test(model, DataLoader(test_dataset, batch_size=256, num_workers=num_cores, pin_memory=True, shuffle=False))
print(best_test_acc)

0.8892675606331994

In [ ]:
train(model, train_dataset, test_dataset, epochs=10, batch_size=512, lr=0.001, patience=5)

In [ ]:
w1, w2, t = zip(test_dataset[20], test_dataset[10])
w1 = torch.stack(w1).to(device)
w2 = torch.stack(w2).to(device)
t = torch.stack(t).to(device)
print(w1, w2, t)
model.eval()
with torch.no_grad():
    print(model(w1, w2))

In [ ]:
# !pip install onnx
model.eval()
torch.onnx.export(
    model,
    (w1, w2),
    "model.onnx",
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input1', 'input2'],
    output_names=['cmp', 'cmpRev'],
    dynamic_axes={
        'input1': {0: 'batch_size', 1: 'seq_len'},
        'input2': {0: 'batch_size', 1: 'seq_len'},
        'cmp': {0: 'batch_size'},
        'cmpRev': {0: 'batch_size'}
    }
)

In [ ]:
# !pip install onnxruntime
# !pip install onnx-simplifier
!python -m onnxsim model.onnx model_simplified.onnx --no-large-tensor

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                 ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Concat          │ 8              │ 8                │
│ Constant        │ 47             │ 21               │
│ ConstantOfShape │ 2              │ 2                │
│ Gather          │ 8              │ 8                │
│ Gemm            │ 6              │ 6                │
│ Identity        │ 6              │ 0                │
│ LSTM            │ 4              │ 4                │
│ Mul             │ 1              │ 1                │
│ Neg             │ 1              │ 1                │
│ Relu            │ 2              │ 2                │
│ Reshape         │ 2              │ 2                │
│ Shape           │ 2              │ 2                │
│ Sigmoid         │ 2              │ 2                │
│ Slice           │ 8              │ 8                │
│

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession("model_simplified.onnx")

w1 = CharPairwiseDataset.tokenize(split_identifier('key'), 25).unsqueeze(0).numpy()
w2 = CharPairwiseDataset.tokenize(split_identifier('ref'), 25).unsqueeze(0).numpy()

session.run(None, {'input1': w1, 'input2': w2})

[array(0.01904154, dtype=float32), array(0.9874006, dtype=float32)]